# GPU V3 — Matrix NMS: Kiểm tra đúng/sai và benchmark trên Colab

Notebook này chạy implementation của repo. Matrix NMS là soft suppression, nên được kiểm tra so với `matrix_nms_reference`, không so với output của greedy NMS.

Trên Colab chọn **Runtime → Change runtime type → T4 GPU**, sau đó chạy lần lượt từng cell từ trên xuống.

In [ ]:
!git clone --depth 1 https://github.com/liltommy142/cuda-nms-numba.git
%cd cuda-nms-numba
!nvidia-smi

import sys
import numba
import numpy as np
print('numba', numba.__version__, '| numpy', np.__version__)

from numba import cuda
assert cuda.is_available(), 'Enable a CUDA GPU runtime in Colab, then reconnect.'
print('GPU:', cuda.get_current_device().name)

sys.path.insert(0, 'src')

# Smoke test before anything else: fail here with a clear message instead of
# halfway through a benchmark. V3's kernels use grid-stride loops plus the
# scalar args n/method/sigma -- the pattern that failed to compile on numba
# 0.66 (see requirements.txt). This checks that they compile and return usable
# indices; the pytest cell below is what verifies them against the CPU oracle.
from cpu_baseline import load_data
from gpu_v3 import run_gpu_v3_matrix_nms

_boxes, _scores = load_data(32, seed=0)
try:
    _keep = run_gpu_v3_matrix_nms(_boxes, _scores, score_threshold=0.05, method='gaussian', sigma=2.0)
except Exception as exc:
    print('Preinstalled numba failed to compile/run the V3 Matrix NMS kernels:', type(exc).__name__, exc)
    print('Fallback: run  !pip install -q "numba==0.59.1" "numpy<2"  then')
    print('Runtime > Restart session, and re-run this cell.')
    raise
assert _keep.ndim == 1 and len(_keep) > 0 and int(_keep.max()) < len(_boxes), \
    'V3 kernels ran but returned invalid indices - a correctness bug, not a numba incompatibility'
print('JIT smoke test OK on numba', numba.__version__, '- V3 Matrix NMS kernels (loops + scalar args)')


## Kiểm tra đúng/sai (Correctness)

Các test đầu tiên kiểm chứng CPU Matrix-NMS oracle; sau đó test GPU so sánh cả output linear lẫn Gaussian của V3 với oracle đó.

In [ ]:
!pytest tests/test_correctness.py -v -k 'matrix_nms or gpu_v3'


## Benchmark lặp lại

Speedup bên dưới là so sánh hiệu năng với greedy CPU NMS, không phải khẳng định V3 cho ra detection giống hệt.

In [ ]:
!python benchmarks/run_all.py --versions cpu v3 --n 100 1000 10000 --warmup 2 --repeats 7 --json benchmarks/results/colab_t4_v3.json
from google.colab import files
files.download('benchmarks/results/colab_t4_v3.json')
